# SciGraph Project Walkthrough

This notebook is the easiest entry point for reviewing the project. It does not replace the reproducible command-line pipeline or the full written report; it gives a guided tour through the generated tables, reports, and figures.

Current run: **2,500 OpenAlex article records**, processed through ingestion, data quality, Spark Bronze/Silver/Gold tables, text analytics, clustering, graph analysis, temporal analysis, streaming simulation, retrieval, ML, ranking, feature mart generation, and visualizations.

## 1. Setup

Run this notebook from either the project root or the `notebooks/` folder. It reads generated artifacts from `reports/`, `docs/`, and `data/`; it does not call the OpenAlex API or rerun Spark.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DOCS = ROOT / 'docs'
RESULTS = ROOT / 'reports' / 'results'
TABLES = ROOT / 'reports' / 'tables'
FIGURES = ROOT / 'reports' / 'figures'

def load_json(name):
    return json.loads((RESULTS / name).read_text(encoding='utf-8'))

milestones = pd.read_csv(TABLES / 'milestone_summary.csv')
retrieval = pd.read_csv(TABLES / 'retrieval_ablation_summary.csv')

available = {
    'project_root': ROOT,
    'tables': sorted(path.name for path in TABLES.glob('*.csv')),
    'figures': sorted(path.name for path in FIGURES.glob('*.png')),
}
available

## 2. Project Map

The project is organized as a pipeline. Raw OpenAlex records are preserved, Spark creates normalized Bronze/Silver/Gold tables, then downstream analysis stages write JSON reports, Markdown docs, Parquet outputs, CSV summaries, and PNG figures.

In [ ]:
milestones

## 3. High-Level Run Metrics

This cell pulls the most important numbers from the generated JSON reports. These values should match `docs/PROJECT_REPORT.md` and the CSV tables under `reports/tables/`.

In [ ]:
schema = load_json('schema_summary.json')
pipeline = load_json('bronze_silver_gold_report.json')
text = load_json('text_analysis_report.json')
cluster = load_json('clustering_report.json')
citation = load_json('citation_graph_report.json')
author = load_json('author_collaboration_report.json')
temporal = load_json('temporal_analysis_report.json')
streaming = load_json('streaming_simulation_report.json')
ml = load_json('ml_citation_prediction_report.json')
feature = load_json('feature_mart_report.json')

summary = pd.DataFrame([
    ('Raw records', schema['dataset_statistics']['record_count']),
    ('Unique authors', schema['dataset_statistics']['unique_authors']),
    ('Gold publications', pipeline['stage_counts']['gold_publications']),
    ('Gold citation edges', pipeline['stage_counts']['gold_citation_edges']),
    ('Text documents', text['document_count']),
    ('Vocabulary size', text['vocabulary_size']),
    ('Selected K', cluster['selected_k']),
    ('Author collaboration edges', author['collaboration_edge_count']),
    ('Topic-year rows', temporal['topic_year_row_count']),
    ('Streaming alerts', streaming['alert_count']),
    ('ML AUC', round(ml['area_under_roc'], 4)),
    ('Feature mart rows', feature['publication_rows']),
], columns=['metric', 'value'])
summary

## 4. Dataset, Quality, And Pipeline Figures

These figures show the sample distribution, quality-check outcomes, and row counts across the Spark pipeline.

In [ ]:
for name in [
    'dataset_publications_by_year.png',
    'data_quality_checks.png',
    'pipeline_stage_counts.png',
]:
    display(Markdown(f'### {name}'))
    display(Image(filename=str(FIGURES / name)))

## 5. Text And Clustering

Text analysis reconstructs abstracts from OpenAlex inverted indexes, filters to English-language records, computes terms and TF-IDF features, and clusters documents with Spark ML K-means.

In [ ]:
for name in ['text_top_terms.png', 'clustering_results.png']:
    display(Markdown(f'### {name}'))
    display(Image(filename=str(FIGURES / name)))

## 6. Graph And Temporal Analysis

The citation graph still has an important negative result: the sampled publications do not cite each other inside the sample. The author collaboration graph is more informative because coauthorships are directly available in each sampled record.

In [ ]:
for name in [
    'citation_graph_summary.png',
    'author_collaboration_summary.png',
    'temporal_yearly_metrics.png',
]:
    display(Markdown(f'### {name}'))
    display(Image(filename=str(FIGURES / name)))

## 7. Streaming, Retrieval, ML, Ranking, And Feature Mart

These are the final analysis layers: simulated micro-batch monitoring, retrieval-system comparison, citation-outcome prediction, composite ranking, and integrated feature availability.

In [ ]:
retrieval.sort_values('ndcg_at_10', ascending=False)

In [ ]:
for name in [
    'streaming_batch_metrics.png',
    'retrieval_ablation.png',
    'ml_citation_prediction.png',
    'ranking_top_publications.png',
    'feature_mart_availability.png',
]:
    display(Markdown(f'### {name}'))
    display(Image(filename=str(FIGURES / name)))

## 8. Where To Read More

- Full final report: `docs/PROJECT_REPORT.md`
- Visualization gallery: `docs/VISUALIZATIONS.md`
- Data dictionary: `docs/DATA_DICTIONARY.md`
- Retrieval details: `docs/RETRIEVAL.md` and `docs/ADVANCED_RETRIEVAL.md`
- Feature mart details: `docs/FEATURE_MART.md`
- Error analysis: `docs/ERROR_ANALYSIS.md`

To regenerate the current artifacts from the project root, run the commands in the Reproducibility section of `docs/PROJECT_REPORT.md`.